# Gold Layer Setup

**Purpose**: Creates the Gold schema and all dimension and fact table structures for the star-schema analytics layer.

**Run this notebook once** (or whenever you need to recreate the schema). All statements are idempotent (`CREATE TABLE IF NOT EXISTS`).

## What this notebook does
1. Creates the `GOLD` schema
2. Creates 12 dimension table structures
3. Creates 10 fact table structures  
4. Seeds **DIM_DATE** (generated calendar dimension — no Silver dependency)
5. Seeds **DIM_CHANNEL** (4 hardcoded sales channel rows)
6. Prints a validation summary of all tables

## Run order
```
GOLD_SETUP.ipynb  → GOLD_INCREMENTAL_LOAD.ipynb (run on schedule)
```

## Dependencies
- Active Snowflake session (Snowpark)
- `SILVER` schema must exist with data loaded before running incremental load
- No dependencies for this setup notebook (schema + tables only)

In [ ]:
# ─────────────────────────────────────────────
# Cell 1 — Imports and Session
# ─────────────────────────────────────────────
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, lit, current_timestamp

session = get_active_session()

# ─── Constants ────────────────────────────────
GOLD_SCHEMA   = 'GOLD'
SILVER_SCHEMA = 'SILVER'

print(f'Session: {session.get_current_database()}.{session.get_current_schema()}')
print(f'Gold target schema : {GOLD_SCHEMA}')
print(f'Silver source schema: {SILVER_SCHEMA}')

---
## Schema Setup

In [ ]:
# ─────────────────────────────────────────────
# Cell 2 — Create GOLD Schema
# ─────────────────────────────────────────────
session.sql(f'CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}').collect()
print(f'✓ Schema {GOLD_SCHEMA} ready')

---
## Dimension Tables

All 12 Gold dimensions are created with `IF NOT EXISTS`, so re-running is safe.  
Tables that require Silver data are created empty here and populated by **GOLD_INCREMENTAL_LOAD.ipynb**.  
**DIM_DATE** and **DIM_CHANNEL** are seeded immediately in the cells below.

In [ ]:
# ─────────────────────────────────────────────
# Cell 3 — DIM_DATE
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.DIM_DATE (
    DATE_KEY         INTEGER       NOT NULL  COMMENT 'YYYYMMDD integer key',
    FULL_DATE        DATE          NOT NULL,
    DAY_OF_WEEK      INTEGER,      -- 1=Sunday … 7=Saturday (Snowflake default)
    DAY_NAME         VARCHAR(10),
    DAY_OF_MONTH     INTEGER,
    DAY_OF_YEAR      INTEGER,
    WEEK_OF_YEAR     INTEGER,
    MONTH            INTEGER,
    MONTH_NAME       VARCHAR(10),
    QUARTER          INTEGER,
    YEAR             INTEGER,
    IS_WEEKEND       BOOLEAN,
    IS_HOLIDAY       BOOLEAN       DEFAULT FALSE,
    HOLIDAY_NAME     VARCHAR(100),
    FISCAL_YEAR      INTEGER,
    FISCAL_QUARTER   INTEGER,
    FISCAL_MONTH     INTEGER,
    CONSTRAINT PK_DIM_DATE PRIMARY KEY (DATE_KEY)
)
''').collect()
print('✓ DIM_DATE created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 4 — Populate DIM_DATE
# Uses Snowflake GENERATOR to create one row per calendar day.
# Date range: 1 year before earliest Silver order → 2 years after latest.
# ─────────────────────────────────────────────
existing = session.table(f'{GOLD_SCHEMA}.DIM_DATE').count()
if existing > 0:
    print(f'DIM_DATE already contains {existing:,} rows — skipping regeneration.')
    print('To regenerate, run: TRUNCATE TABLE GOLD.DIM_DATE  then re-run this cell.')
else:
    session.sql(f'''
    INSERT INTO {GOLD_SCHEMA}.DIM_DATE
    WITH date_bounds AS (
        SELECT
            DATEADD(year, -1, LEAST(
                MIN(wb.min_dt), MIN(mb.min_dt), MIN(wlb.min_dt)
            ))::DATE AS start_date,
            DATEADD(year, 2, GREATEST(
                MAX(wb.max_dt), MAX(mb.max_dt), MAX(wlb.max_dt)
            ))::DATE AS end_date
        FROM
            (SELECT MIN(ORDER_DATE) AS min_dt, MAX(ORDER_DATE) AS max_dt
             FROM {SILVER_SCHEMA}.WEB_ORDERS) wb,
            (SELECT MIN(ORDER_DATE) AS min_dt, MAX(ORDER_DATE) AS max_dt
             FROM {SILVER_SCHEMA}.MOB_ORDERS) mb,
            (SELECT MIN(ORDER_DATE) AS min_dt, MAX(ORDER_DATE) AS max_dt
             FROM {SILVER_SCHEMA}.WHL_ORDERS) wlb
    ),
    date_spine AS (
        SELECT DATEADD(day, SEQ4(), b.start_date)::DATE AS dt
        FROM   date_bounds b,
               TABLE(GENERATOR(ROWCOUNT => 6000))
        WHERE  DATEADD(day, SEQ4(), b.start_date) <= b.end_date
    )
    SELECT
        TO_NUMBER(TO_VARCHAR(dt, 'YYYYMMDD'))  AS DATE_KEY,
        dt                                     AS FULL_DATE,
        DAYOFWEEK(dt)                          AS DAY_OF_WEEK,
        DAYNAME(dt)                            AS DAY_NAME,
        DAY(dt)                                AS DAY_OF_MONTH,
        DAYOFYEAR(dt)                          AS DAY_OF_YEAR,
        WEEKOFYEAR(dt)                         AS WEEK_OF_YEAR,
        MONTH(dt)                              AS MONTH,
        MONTHNAME(dt)                          AS MONTH_NAME,
        QUARTER(dt)                            AS QUARTER,
        YEAR(dt)                               AS YEAR,
        IFF(DAYOFWEEK(dt) IN (1, 7), TRUE, FALSE) AS IS_WEEKEND,
        FALSE                                  AS IS_HOLIDAY,
        NULL                                   AS HOLIDAY_NAME,
        YEAR(dt)                               AS FISCAL_YEAR,
        QUARTER(dt)                            AS FISCAL_QUARTER,
        MONTH(dt)                              AS FISCAL_MONTH
    FROM date_spine
    ''').collect()

    count = session.table(f'{GOLD_SCHEMA}.DIM_DATE').count()
    print(f'✓ DIM_DATE populated: {count:,} rows')
    dates = session.sql(f'SELECT MIN(FULL_DATE), MAX(FULL_DATE) FROM {GOLD_SCHEMA}.DIM_DATE').collect()
    print(f'  Date range: {dates[0][0]} → {dates[0][1]}')

In [ ]:
# ─────────────────────────────────────────────
# Cell 5 — DIM_CHANNEL
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.DIM_CHANNEL (
    CHANNEL_KEY    INTEGER       NOT NULL COMMENT 'Surrogate key',
    CHANNEL_ID     VARCHAR(20)   NOT NULL COMMENT 'Business key: web/mobile/wholesale/marketplace',
    CHANNEL_NAME   VARCHAR(50)   NOT NULL,
    CHANNEL_TYPE   VARCHAR(10)   NOT NULL COMMENT 'B2C or B2B',
    IS_ACTIVE      BOOLEAN       DEFAULT TRUE,
    CONSTRAINT PK_DIM_CHANNEL PRIMARY KEY (CHANNEL_KEY)
)
''').collect()
print('✓ DIM_CHANNEL created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 6 — Populate DIM_CHANNEL (4 hardcoded rows)
# ─────────────────────────────────────────────
existing = session.table(f'{GOLD_SCHEMA}.DIM_CHANNEL').count()
if existing > 0:
    print(f'DIM_CHANNEL already contains {existing} rows — skipping.')
else:
    session.sql(f'''
    INSERT INTO {GOLD_SCHEMA}.DIM_CHANNEL (CHANNEL_KEY, CHANNEL_ID, CHANNEL_NAME, CHANNEL_TYPE, IS_ACTIVE) VALUES
        (1, 'web',         'Web Store',        'B2C', TRUE),
        (2, 'mobile',      'Mobile App',       'B2C', TRUE),
        (3, 'wholesale',   'Wholesale Portal', 'B2B', TRUE),
        (4, 'marketplace', 'Marketplace',      'B2C', TRUE)
    ''').collect()
    print(f'✓ DIM_CHANNEL populated: 4 rows (web, mobile, wholesale, marketplace)')

In [ ]:
# ─────────────────────────────────────────────
# Cell 7 — DIM_CUSTOMER  (SCD Type 2)
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.DIM_CUSTOMER (
    CUSTOMER_KEY      INTEGER        NOT NULL  COMMENT 'Surrogate key',
    CUSTOMER_ID       VARCHAR(50)    NOT NULL  COMMENT 'Business key = BRIDGE_CUSTOMER_IDENTITY.CANONICAL_CUSTOMER_ID',
    EMAIL_ADDRESS     VARCHAR(255),
    FIRST_NAME        VARCHAR(100),
    LAST_NAME         VARCHAR(100),
    FULL_NAME         VARCHAR(200),
    PHONE_NUMBER      VARCHAR(50),
    CUSTOMER_TYPE     VARCHAR(10)    COMMENT 'B2C or B2B',
    COMPANY_NAME      VARCHAR(200)   COMMENT 'Populated for B2B wholesale customers',
    COUNTRY_CODE      VARCHAR(10),
    CITY_NAME         VARCHAR(100),
    POSTAL_CODE       VARCHAR(20),
    CURRENCY_CODE     VARCHAR(5),
    REGISTRATION_DATE DATE,
    LOYALTY_TIER      VARCHAR(50),
    LOYALTY_TIER_RANK INTEGER,
    ACCOUNT_STATUS    VARCHAR(30),
    MARKETING_OPT_IN  BOOLEAN,
    PRIMARY_CHANNEL   VARCHAR(20),
    CHANNELS_USED     VARCHAR(100)   COMMENT 'Comma-separated list of channels',
    EFFECTIVE_DATE    DATE           NOT NULL  COMMENT 'SCD2: row valid from this date',
    VALID_TO          DATE           COMMENT 'SCD2: row valid until this date, NULL = current row',
    IS_CURRENT        BOOLEAN        NOT NULL  DEFAULT TRUE,
    CONSTRAINT PK_DIM_CUSTOMER PRIMARY KEY (CUSTOMER_KEY)
)
''').collect()
print('✓ DIM_CUSTOMER created (SCD Type 2)')

In [ ]:
# ─────────────────────────────────────────────
# Cell 8 — DIM_PRODUCT  (SCD Type 2)
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.DIM_PRODUCT (
    PRODUCT_KEY            INTEGER        NOT NULL  COMMENT 'Surrogate key',
    PRODUCT_SKU            VARCHAR(50)    NOT NULL  COMMENT 'Business key',
    PRODUCT_NAME           VARCHAR(255),
    PRIMARY_CATEGORY_CODE  VARCHAR(50),
    PRIMARY_CATEGORY_NAME  VARCHAR(100),
    ALL_CATEGORIES         VARCHAR(500)   COMMENT 'Comma-separated canonical category names',
    CURRENT_UNIT_PRICE     DECIMAL(12,2)  COMMENT 'Type 1: overwritten each load',
    AVERAGE_UNIT_PRICE     DECIMAL(12,2)  COMMENT 'Type 1: overwritten each load',
    MIN_UNIT_PRICE         DECIMAL(12,2)  COMMENT 'Type 1: overwritten each load',
    MAX_UNIT_PRICE         DECIMAL(12,2)  COMMENT 'Type 1: overwritten each load',
    PRODUCT_STATUS         VARCHAR(20)    COMMENT 'Active or Discontinued — SCD2 tracked',
    FIRST_SALE_DATE        DATE,
    LAST_SALE_DATE         DATE,
    EFFECTIVE_DATE         DATE           NOT NULL  COMMENT 'SCD2: row valid from this date',
    VALID_TO               DATE           COMMENT 'SCD2: row valid until this date, NULL = current row',
    IS_CURRENT             BOOLEAN        NOT NULL  DEFAULT TRUE,
    CONSTRAINT PK_DIM_PRODUCT PRIMARY KEY (PRODUCT_KEY)
)
''').collect()
print('✓ DIM_PRODUCT created (SCD Type 2)')

In [ ]:
# ─────────────────────────────────────────────
# Cell 9 — DIM_CATEGORY
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.DIM_CATEGORY (
    CATEGORY_KEY   INTEGER      NOT NULL  COMMENT 'Surrogate key',
    CATEGORY_CODE  VARCHAR(50)  NOT NULL  COMMENT 'Business key',
    CATEGORY_NAME  VARCHAR(100),
    IS_ACTIVE      BOOLEAN      DEFAULT TRUE,
    CONSTRAINT PK_DIM_CATEGORY PRIMARY KEY (CATEGORY_KEY)
)
''').collect()
print('✓ DIM_CATEGORY created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 10 — DIM_MARKETING_CHANNEL
# Distinct from DIM_CHANNEL: how customers were acquired (email/social/search/display)
# versus where they purchase (web/mobile/wholesale/marketplace)
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.DIM_MARKETING_CHANNEL (
    MARKETING_CHANNEL_KEY    INTEGER      NOT NULL  COMMENT 'Surrogate key',
    MARKETING_CHANNEL_ID     VARCHAR(50)  NOT NULL  COMMENT 'Business key (canonical channel name)',
    MARKETING_CHANNEL_NAME   VARCHAR(100),
    IS_PAID                  BOOLEAN      COMMENT 'True for paid placements (search/display/paid social)',
    IS_ACTIVE                BOOLEAN      DEFAULT TRUE,
    CONSTRAINT PK_DIM_MARKETING_CHANNEL PRIMARY KEY (MARKETING_CHANNEL_KEY)
)
''').collect()
print('✓ DIM_MARKETING_CHANNEL created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 11 — DIM_PAYMENT_METHOD
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.DIM_PAYMENT_METHOD (
    PAYMENT_METHOD_KEY   INTEGER      NOT NULL  COMMENT 'Surrogate key',
    PAYMENT_METHOD_CODE  VARCHAR(50)  NOT NULL  COMMENT 'Business key',
    PAYMENT_METHOD_NAME  VARCHAR(100),
    CATEGORY             VARCHAR(50)  COMMENT 'card / digital_wallet / bank_transfer / cash',
    IS_ACTIVE            BOOLEAN      DEFAULT TRUE,
    CONSTRAINT PK_DIM_PAYMENT_METHOD PRIMARY KEY (PAYMENT_METHOD_KEY)
)
''').collect()
print('✓ DIM_PAYMENT_METHOD created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 12 — DIM_ORDER_STATUS
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.DIM_ORDER_STATUS (
    ORDER_STATUS_KEY   INTEGER      NOT NULL  COMMENT 'Surrogate key',
    STATUS_CODE        VARCHAR(30)  NOT NULL  COMMENT 'Business key (canonical value)',
    STATUS_NAME        VARCHAR(50),
    STATUS_DESCRIPTION VARCHAR(200),
    STATUS_CATEGORY    VARCHAR(20)  COMMENT 'Open / Closed / Cancelled',
    IS_ACTIVE          BOOLEAN      DEFAULT TRUE,
    CONSTRAINT PK_DIM_ORDER_STATUS PRIMARY KEY (ORDER_STATUS_KEY)
)
''').collect()
print('✓ DIM_ORDER_STATUS created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 13 — DIM_PAYMENT_STATUS
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.DIM_PAYMENT_STATUS (
    PAYMENT_STATUS_KEY  INTEGER      NOT NULL  COMMENT 'Surrogate key',
    STATUS_CODE         VARCHAR(30)  NOT NULL  COMMENT 'Business key',
    STATUS_NAME         VARCHAR(50),
    IS_SUCCESSFUL       BOOLEAN,
    IS_ACTIVE           BOOLEAN      DEFAULT TRUE,
    CONSTRAINT PK_DIM_PAYMENT_STATUS PRIMARY KEY (PAYMENT_STATUS_KEY)
)
''').collect()
print('✓ DIM_PAYMENT_STATUS created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 14 — DIM_LOYALTY_TIER
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.DIM_LOYALTY_TIER (
    LOYALTY_TIER_KEY  INTEGER      NOT NULL  COMMENT 'Surrogate key',
    TIER_CODE         VARCHAR(30)  NOT NULL  COMMENT 'Business key (canonical tier name)',
    TIER_NAME         VARCHAR(50),
    TIER_RANK         INTEGER      COMMENT '1=standard, 2=silver, 3=gold, 4=platinum',
    IS_ACTIVE         BOOLEAN      DEFAULT TRUE,
    CONSTRAINT PK_DIM_LOYALTY_TIER PRIMARY KEY (LOYALTY_TIER_KEY)
)
''').collect()
print('✓ DIM_LOYALTY_TIER created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 15 — DIM_PAYMENT_TERMS  (wholesale B2B only)
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.DIM_PAYMENT_TERMS (
    PAYMENT_TERMS_KEY   INTEGER      NOT NULL  COMMENT 'Surrogate key',
    PAYMENT_TERMS_CODE  VARCHAR(20)  NOT NULL  COMMENT 'Business key',
    PAYMENT_TERMS_NAME  VARCHAR(50),
    DAYS_TO_PAY         INTEGER      COMMENT 'Derived: 30 for NET30, 60 for NET60, 0 for COD/PREPAID',
    IS_ACTIVE           BOOLEAN      DEFAULT TRUE,
    CONSTRAINT PK_DIM_PAYMENT_TERMS PRIMARY KEY (PAYMENT_TERMS_KEY)
)
''').collect()
print('✓ DIM_PAYMENT_TERMS created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 16 — DIM_CAMPAIGN  (SCD Type 2)
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.DIM_CAMPAIGN (
    CAMPAIGN_KEY              INTEGER      NOT NULL  COMMENT 'Surrogate key',
    CAMPAIGN_ID               INTEGER      NOT NULL  COMMENT 'Business key from MKTG_CAMPAIGNS',
    CAMPAIGN_NAME             VARCHAR(255),
    MARKETING_CHANNEL_KEY     INTEGER      COMMENT 'FK → DIM_MARKETING_CHANNEL',
    TARGET_SEGMENT            VARCHAR(100),
    START_DATE                DATE,
    END_DATE                  DATE,
    CAMPAIGN_STATUS           VARCHAR(30),
    EFFECTIVE_DATE            DATE         NOT NULL  COMMENT 'SCD2: row valid from this date',
    VALID_TO                  DATE         COMMENT 'SCD2: NULL = current row',
    IS_CURRENT                BOOLEAN      NOT NULL  DEFAULT TRUE,
    CONSTRAINT PK_DIM_CAMPAIGN PRIMARY KEY (CAMPAIGN_KEY)
)
''').collect()
print('✓ DIM_CAMPAIGN created (SCD Type 2)')

In [ ]:
# ─────────────────────────────────────────────
# Cell 17 — Validate all dimension tables exist
# ─────────────────────────────────────────────
dim_tables = [
    'DIM_DATE', 'DIM_CHANNEL', 'DIM_CUSTOMER', 'DIM_PRODUCT',
    'DIM_CATEGORY', 'DIM_MARKETING_CHANNEL', 'DIM_PAYMENT_METHOD',
    'DIM_ORDER_STATUS', 'DIM_PAYMENT_STATUS', 'DIM_LOYALTY_TIER',
    'DIM_PAYMENT_TERMS', 'DIM_CAMPAIGN'
]
print('Dimension table row counts:')
for t in dim_tables:
    try:
        cnt = session.table(f'{GOLD_SCHEMA}.{t}').count()
        print(f'  {t:<30} {cnt:>8,} rows')
    except Exception as e:
        print(f'  {t:<30}  ERROR: {e}')

---
## Fact Tables

All 10 fact tables are created empty here. Data is populated by **GOLD_INCREMENTAL_LOAD.ipynb**.  
Fact tables depend on all dimension tables being populated first.

In [ ]:
# ─────────────────────────────────────────────
# Cell 18 — FACT_ORDERS  (core — one row per order, all channels)
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.FACT_ORDERS (
    ORDER_KEY          INTEGER        NOT NULL  COMMENT 'Surrogate key',
    ORDER_ID           INTEGER        NOT NULL  COMMENT 'Degenerate dimension — business order ID',
    CUSTOMER_KEY       INTEGER        COMMENT 'FK → DIM_CUSTOMER; NULL for marketplace (Amazon masks buyer email)',
    ORDER_DATE_KEY     INTEGER        COMMENT 'FK → DIM_DATE',
    CHANNEL_KEY        INTEGER        COMMENT 'FK → DIM_CHANNEL',
    ORDER_STATUS_KEY   INTEGER        COMMENT 'FK → DIM_ORDER_STATUS',
    ORDER_DATE         DATE,
    ORDER_AMOUNT       DECIMAL(14,2),
    ORDER_AMOUNT_USD   DECIMAL(14,2)  COMMENT 'ORDER_AMOUNT * EXCHANGE_RATE',
    EXCHANGE_RATE      DECIMAL(10,6)  DEFAULT 1.0  COMMENT '1.0 for USD; placeholder for future FX integration',
    TAX_AMOUNT         DECIMAL(14,2)  COMMENT 'Populated for wholesale orders only',
    CURRENCY_CODE      VARCHAR(5),
    PROMO_CODE         VARCHAR(100),
    _SILVER_LOAD_TIMESTAMP  TIMESTAMP_NTZ  COMMENT 'Inherited from Silver source — used as incremental watermark',
    CONSTRAINT PK_FACT_ORDERS PRIMARY KEY (ORDER_KEY)
)
''').collect()
print('✓ FACT_ORDERS created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 19 — FACT_ORDERS_MOBILE_EXT
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.FACT_ORDERS_MOBILE_EXT (
    ORDER_KEY        INTEGER      NOT NULL  COMMENT 'PK + FK → FACT_ORDERS',
    PLATFORM_TYPE    VARCHAR(20)  COMMENT 'Android or iOS',
    SOURCE_ORDER_ID  VARCHAR(50)  COMMENT 'Degenerate dimension — mobile app internal order ID',
    CONSTRAINT PK_FACT_ORDERS_MOBILE_EXT PRIMARY KEY (ORDER_KEY)
)
''').collect()
print('✓ FACT_ORDERS_MOBILE_EXT created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 20 — FACT_ORDERS_WHOLESALE_EXT
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.FACT_ORDERS_WHOLESALE_EXT (
    ORDER_KEY              INTEGER       NOT NULL  COMMENT 'PK + FK → FACT_ORDERS',
    BUYER_PO_NUMBER        VARCHAR(100)  COMMENT 'Degenerate dimension — customer purchase order number',
    PAYMENT_TERMS_KEY      INTEGER       COMMENT 'FK → DIM_PAYMENT_TERMS',
    ORDER_AMOUNT_EXCL_TAX  DECIMAL(14,2),
    CONSTRAINT PK_FACT_ORDERS_WHOLESALE_EXT PRIMARY KEY (ORDER_KEY)
)
''').collect()
print('✓ FACT_ORDERS_WHOLESALE_EXT created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 21 — FACT_ORDERS_MARKETPLACE_EXT
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.FACT_ORDERS_MARKETPLACE_EXT (
    ORDER_KEY            INTEGER      NOT NULL  COMMENT 'PK + FK → FACT_ORDERS',
    AMAZON_ORDER_ID      VARCHAR(50)  COMMENT 'Degenerate dimension — Amazon order identifier',
    FULFILLMENT_CHANNEL  VARCHAR(20)  COMMENT 'FBA, FBM, etc.',
    CONSTRAINT PK_FACT_ORDERS_MARKETPLACE_EXT PRIMARY KEY (ORDER_KEY)
)
''').collect()
print('✓ FACT_ORDERS_MARKETPLACE_EXT created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 22 — FACT_ORDER_ITEMS  (unified line items, all channels)
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.FACT_ORDER_ITEMS (
    ORDER_ITEM_KEY    INTEGER        NOT NULL  COMMENT 'Surrogate key',
    ORDER_ID          INTEGER        COMMENT 'Degenerate dimension',
    ORDER_KEY         INTEGER        COMMENT 'FK → FACT_ORDERS',
    PRODUCT_KEY       INTEGER        COMMENT 'FK → DIM_PRODUCT',
    CUSTOMER_KEY      INTEGER        COMMENT 'FK → DIM_CUSTOMER (inherited from FACT_ORDERS)',
    ORDER_DATE_KEY    INTEGER        COMMENT 'FK → DIM_DATE (inherited from FACT_ORDERS)',
    CHANNEL_KEY       INTEGER        COMMENT 'FK → DIM_CHANNEL',
    CATEGORY_KEY      INTEGER        COMMENT 'FK → DIM_CATEGORY',
    QUANTITY          INTEGER,
    UNIT_PRICE        DECIMAL(12,2),
    LINE_TOTAL        DECIMAL(14,2),
    IS_RETURNED       BOOLEAN        COMMENT 'Web orders only; NULL for other channels',
    MARKETPLACE_FEE   DECIMAL(10,2)  COMMENT 'Marketplace orders only; NULL for other channels',
    _SILVER_LOAD_TIMESTAMP  TIMESTAMP_NTZ,
    CONSTRAINT PK_FACT_ORDER_ITEMS PRIMARY KEY (ORDER_ITEM_KEY)
)
''').collect()
print('✓ FACT_ORDER_ITEMS created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 23 — FACT_PAYMENTS
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.FACT_PAYMENTS (
    PAYMENT_KEY            INTEGER        NOT NULL  COMMENT 'Surrogate key',
    TRANSACTION_ID         VARCHAR(100)   COMMENT 'Degenerate dimension',
    ORDER_ID               INTEGER        COMMENT 'Degenerate dimension',
    ORDER_KEY              INTEGER        COMMENT 'FK → FACT_ORDERS',
    PAYMENT_METHOD_KEY     INTEGER        COMMENT 'FK → DIM_PAYMENT_METHOD',
    PAYMENT_STATUS_KEY     INTEGER        COMMENT 'FK → DIM_PAYMENT_STATUS',
    TRANSACTION_DATE_KEY   INTEGER        COMMENT 'FK → DIM_DATE',
    CHANNEL_KEY            INTEGER        COMMENT 'FK → DIM_CHANNEL',
    PROCESSED_DATETIME     TIMESTAMP_NTZ,
    GROSS_AMOUNT           DECIMAL(14,2),
    FEE_AMOUNT             DECIMAL(10,2),
    NET_AMOUNT             DECIMAL(14,2),
    CURRENCY_CODE          VARCHAR(5),
    FAILURE_REASON_CODE    VARCHAR(50),
    IS_FIRST_TRANSACTION   BOOLEAN,
    GATEWAY_REGION         VARCHAR(50),
    _SILVER_LOAD_TIMESTAMP TIMESTAMP_NTZ,
    CONSTRAINT PK_FACT_PAYMENTS PRIMARY KEY (PAYMENT_KEY)
)
''').collect()
print('✓ FACT_PAYMENTS created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 24 — FACT_SHIPMENTS
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.FACT_SHIPMENTS (
    SHIPMENT_KEY           INTEGER        NOT NULL  COMMENT 'Surrogate key',
    SHIPMENT_ID            VARCHAR(100)   COMMENT 'Degenerate dimension',
    ORDER_ID               INTEGER        COMMENT 'Degenerate dimension',
    ORDER_KEY              INTEGER        COMMENT 'FK → FACT_ORDERS',
    SHIPMENT_DATE_KEY      INTEGER        COMMENT 'FK → DIM_DATE (pickup date)',
    DELIVERY_DATE_KEY      INTEGER        COMMENT 'FK → DIM_DATE (delivery date; NULL if in transit)',
    CHANNEL_KEY            INTEGER        COMMENT 'FK → DIM_CHANNEL',
    ORIGIN_FACILITY        VARCHAR(100),
    PICKUP_DATETIME        TIMESTAMP_NTZ,
    DELIVERY_DATETIME      TIMESTAMP_NTZ  COMMENT 'NULL for in-transit shipments',
    PACKAGE_WEIGHT_KG      DECIMAL(10,3),
    WEIGHT_UNIT_ORIGINAL   VARCHAR(5),
    SHIPMENT_STATUS        VARCHAR(30),
    SIGNATURE_REQUIRED     BOOLEAN,
    INSURANCE_VALUE        DECIMAL(10,2),
    DAYS_TO_DELIVER        DECIMAL(10,2)  COMMENT 'Derived: DELIVERY_DATETIME - PICKUP_DATETIME in days',
    ON_TIME_DELIVERY_FLAG  BOOLEAN        COMMENT 'Derived: delivered within 7 days baseline',
    _SILVER_LOAD_TIMESTAMP TIMESTAMP_NTZ,
    CONSTRAINT PK_FACT_SHIPMENTS PRIMARY KEY (SHIPMENT_KEY)
)
''').collect()
print('✓ FACT_SHIPMENTS created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 25 — FACT_REVIEWS
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.FACT_REVIEWS (
    REVIEW_KEY             INTEGER       NOT NULL  COMMENT 'Surrogate key',
    REVIEW_ID              INTEGER       COMMENT 'Degenerate dimension',
    PRODUCT_KEY            INTEGER       COMMENT 'FK → DIM_PRODUCT',
    CATEGORY_KEY           INTEGER       COMMENT 'FK → DIM_CATEGORY',
    REVIEW_DATE_KEY        INTEGER       COMMENT 'FK → DIM_DATE',
    REVIEWER_HANDLE        VARCHAR(100)  COMMENT 'Anonymised identifier — cannot link to DIM_CUSTOMER',
    STAR_RATING            INTEGER,
    VERIFIED_PURCHASE      BOOLEAN,
    VERIFIED_SOURCE        VARCHAR(50),
    MODERATION_STATUS      VARCHAR(30),
    HELPFUL_VOTES          INTEGER,
    _SILVER_LOAD_TIMESTAMP TIMESTAMP_NTZ,
    CONSTRAINT PK_FACT_REVIEWS PRIMARY KEY (REVIEW_KEY)
)
''').collect()
print('✓ FACT_REVIEWS created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 26 — FACT_SUPPORT_TICKETS
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.FACT_SUPPORT_TICKETS (
    TICKET_KEY             INTEGER        NOT NULL  COMMENT 'Surrogate key',
    TICKET_ID              INTEGER        COMMENT 'Degenerate dimension',
    ORDER_ID               INTEGER        COMMENT 'Degenerate dimension',
    ORDER_KEY              INTEGER        COMMENT 'FK → FACT_ORDERS',
    CUSTOMER_KEY           INTEGER        COMMENT 'FK → DIM_CUSTOMER; resolved via REQUESTER_EMAIL → DIM_CUSTOMER',
    CREATED_DATE_KEY       INTEGER        COMMENT 'FK → DIM_DATE',
    SOLVED_DATE_KEY        INTEGER        COMMENT 'FK → DIM_DATE; NULL if open',
    CHANNEL_KEY            INTEGER        COMMENT 'FK → DIM_CHANNEL',
    ASSIGNEE               VARCHAR(100),
    TAGS                   VARCHAR(500),
    TICKET_STATUS          VARCHAR(30),
    CREATED_DATETIME       TIMESTAMP_NTZ,
    SOLVED_DATETIME        TIMESTAMP_NTZ  COMMENT 'NULL if ticket not yet solved',
    FIRST_REPLY_HOURS      DECIMAL(10,2),
    CSAT_SCORE             DECIMAL(5,2),
    DAYS_TO_SOLVE          DECIMAL(10,2)  COMMENT 'Derived: SOLVED_DATETIME - CREATED_DATETIME in days',
    IS_SOLVED              BOOLEAN        COMMENT 'Derived: TRUE if TICKET_STATUS = solved',
    _SILVER_LOAD_TIMESTAMP TIMESTAMP_NTZ,
    CONSTRAINT PK_FACT_SUPPORT_TICKETS PRIMARY KEY (TICKET_KEY)
)
''').collect()
print('✓ FACT_SUPPORT_TICKETS created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 27 — FACT_USER_DAILY_ENGAGEMENT
# Aggregated from APP_EVENTS at daily grain per user
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.FACT_USER_DAILY_ENGAGEMENT (
    USER_DAILY_KEY           INTEGER        NOT NULL  COMMENT 'Surrogate key',
    USER_ID                  VARCHAR(50)    COMMENT 'Degenerate dimension — matches MOB_CUSTOMERS.CUSTOMER_ID',
    CUSTOMER_KEY             INTEGER        COMMENT 'FK → DIM_CUSTOMER; NULL for ~15% guest sessions',
    ACTIVITY_DATE_KEY        INTEGER        COMMENT 'FK → DIM_DATE',
    ACTIVITY_DATE            DATE,
    APP_VERSION              VARCHAR(20)    COMMENT 'Most recent version used that day',
    IS_AUTHENTICATED         BOOLEAN        COMMENT 'TRUE if any authenticated event that day',
    IS_VPN_DETECTED          BOOLEAN        COMMENT 'TRUE if any VPN detected that day',
    SESSION_COUNT            INTEGER,
    TOTAL_EVENTS             INTEGER,
    FIRST_EVENT_TIME         TIMESTAMP_NTZ,
    LAST_EVENT_TIME          TIMESTAMP_NTZ,
    ACTIVE_MINUTES           INTEGER        COMMENT 'Minutes from first to last event',
    LOGIN_COUNT              INTEGER,
    PRODUCT_VIEW_COUNT       INTEGER,
    CATEGORY_BROWSE_COUNT    INTEGER,
    SEARCH_COUNT             INTEGER,
    ADD_TO_CART_COUNT        INTEGER,
    CHECKOUT_START_COUNT     INTEGER,
    CHECKOUT_COMPLETE_COUNT  INTEGER,
    WISHLIST_ADD_COUNT       INTEGER,
    UNIQUE_PRODUCTS_VIEWED   INTEGER,
    UNIQUE_CATEGORIES_BROWSED INTEGER,
    _SILVER_LOAD_TIMESTAMP   TIMESTAMP_NTZ,
    CONSTRAINT PK_FACT_USER_DAILY_ENGAGEMENT PRIMARY KEY (USER_DAILY_KEY)
)
''').collect()
print('✓ FACT_USER_DAILY_ENGAGEMENT created')

In [ ]:
# ─────────────────────────────────────────────
# Cell 28 — FACT_CAMPAIGN_DAILY
# ─────────────────────────────────────────────
session.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.FACT_CAMPAIGN_DAILY (
    CAMPAIGN_DAILY_KEY       INTEGER        NOT NULL  COMMENT 'Surrogate key',
    CAMPAIGN_KEY             INTEGER        COMMENT 'FK → DIM_CAMPAIGN',
    DATE_KEY                 INTEGER        COMMENT 'FK → DIM_DATE',
    MARKETING_CHANNEL_KEY    INTEGER        COMMENT 'FK → DIM_MARKETING_CHANNEL',
    DATE                     DATE,
    AMOUNT_SPENT             DECIMAL(12,2),
    IMPRESSIONS              INTEGER,
    CLICKS                   INTEGER,
    CONVERSIONS              INTEGER,
    CTR_PERCENT              DECIMAL(8,4),
    CPC_AMOUNT               DECIMAL(10,4)  COMMENT 'NULL when clicks = 0',
    CPA_AMOUNT               DECIMAL(10,4)  COMMENT 'NULL when conversions = 0',
    _SILVER_LOAD_TIMESTAMP   TIMESTAMP_NTZ,
    CONSTRAINT PK_FACT_CAMPAIGN_DAILY PRIMARY KEY (CAMPAIGN_DAILY_KEY)
)
''').collect()
print('✓ FACT_CAMPAIGN_DAILY created')

---
## Final Validation Summary

In [ ]:
# ─────────────────────────────────────────────
# Cell 29 — Full Validation Summary
# ─────────────────────────────────────────────
all_tables = [
    # Dimensions
    ('DIM', 'DIM_DATE'),
    ('DIM', 'DIM_CHANNEL'),
    ('DIM', 'DIM_CUSTOMER'),
    ('DIM', 'DIM_PRODUCT'),
    ('DIM', 'DIM_CATEGORY'),
    ('DIM', 'DIM_MARKETING_CHANNEL'),
    ('DIM', 'DIM_PAYMENT_METHOD'),
    ('DIM', 'DIM_ORDER_STATUS'),
    ('DIM', 'DIM_PAYMENT_STATUS'),
    ('DIM', 'DIM_LOYALTY_TIER'),
    ('DIM', 'DIM_PAYMENT_TERMS'),
    ('DIM', 'DIM_CAMPAIGN'),
    # Facts
    ('FACT', 'FACT_ORDERS'),
    ('FACT', 'FACT_ORDERS_MOBILE_EXT'),
    ('FACT', 'FACT_ORDERS_WHOLESALE_EXT'),
    ('FACT', 'FACT_ORDERS_MARKETPLACE_EXT'),
    ('FACT', 'FACT_ORDER_ITEMS'),
    ('FACT', 'FACT_PAYMENTS'),
    ('FACT', 'FACT_SHIPMENTS'),
    ('FACT', 'FACT_REVIEWS'),
    ('FACT', 'FACT_SUPPORT_TICKETS'),
    ('FACT', 'FACT_USER_DAILY_ENGAGEMENT'),
    ('FACT', 'FACT_CAMPAIGN_DAILY'),
]

print('='*65)
print(f'  GOLD Schema Setup Summary — {GOLD_SCHEMA}')
print('='*65)
print(f'  {"Type":<6} {"Table Name":<40} {"Rows":>8}')
print('-'*65)

errors = []
for table_type, table_name in all_tables:
    try:
        cnt = session.table(f'{GOLD_SCHEMA}.{table_name}').count()
        note = '  ← seeded' if cnt > 0 else ''
        print(f'  {table_type:<6} {table_name:<40} {cnt:>8,}{note}')
    except Exception as e:
        print(f'  {table_type:<6} {table_name:<40}  !! ERROR: {e}')
        errors.append(table_name)

print('='*65)
if errors:
    print(f'  [!] {len(errors)} table(s) failed: {errors}')
else:
    print(f'  ✓ All {len(all_tables)} tables created successfully')
    print(f'  ✓ DIM_DATE and DIM_CHANNEL are pre-seeded')
    print(f'  ✓ Run GOLD_INCREMENTAL_LOAD.ipynb to populate all other tables')
print('='*65)